In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis (`bumps-dream`): LBCO, HRPT

This tutorial demonstrates a practical two-stage workflow for powder
diffraction analysis with EasyDiffraction.

In the first stage, we run a fast local refinement to obtain a sensible
point estimate and parameter uncertainties. In the second stage, we use
these refined values to define fit bounds and then sample the posterior
distribution with DREAM.

The example uses constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 measured on HRPT at PSI.

The goal is not only to obtain a good fit, but also to answer Bayesian
questions such as:

- Which parameter values are most probable?
- How broad are the credible intervals?
- Which parameters are strongly correlated?
- How much uncertainty propagates into the calculated diffraction
  pattern?

## Import Library

In [2]:
import easydiffraction as ed

## Step 1: Create a Project Container

The project object keeps structures, experiments, fit settings, and
plotting utilities together in a single place. We will build the full
workflow inside this object.

Save the project to a directory early on so that you can easily reload
it later if needed.

In [3]:
project = ed.Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
project.save_as('projects/lbco_hrpt_bumps-dream')

Saving project 📦 'untitled_project' to '../../../projects/lbco_hrpt_bumps-dream'


├── 📄 project.cif


├── 📁 structures/


├── 📁 experiments/


├── 📁 analysis/


│   └── 📄 analysis.cif


└── 📄 summary.cif


## Step 2: Build the Structural Model

We define a simple cubic perovskite model for LBCO. La and Ba share the
same crystallographic site with equal occupancy, while Co and O occupy
the remaining ideal perovskite positions.

In [5]:
project.structures.create(name='lbco')

In [6]:
structure = project.structures['lbco']

In [7]:
structure.space_group.name_h_m = 'P m -3 m'
structure.space_group.it_coordinate_system_code = '1'

In [8]:
structure.cell.length_a = 3.88

The atom-site definitions below form the starting structural model. The
parameters are intentionally reasonable rather than fully optimized,
because the refinement step will improve them.

In [9]:
structure.atom_sites.create(
    label='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_type='Biso',
    adp_iso=0.2190,
)
structure.atom_sites.create(
    label='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_type='Biso',
    adp_iso=1.3916,
)

## Step 3: Define the Diffraction Experiment

Next we download the measured powder pattern, create a neutron powder
experiment, and configure the instrument, profile, background, and
excluded regions.

Download the measured data from the repository. Alternatively, you
could use your own data file by providing the path to it instead of
downloading from the repository.

In [10]:
data_path = ed.download_data(id=3, destination='data')

Getting data...


Data #3: La0.5Ba0.5CoO3, HRPT (PSI), 300 K


✅ Data #3 already present at '../../../data/ed-3.xye'. Keeping existing.


Create the experiment object and specify the sample form, beam mode,
and radiation probe.

In [11]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully


Experiment 🔬 'hrpt'. Number of data points: 3098.


In [12]:
experiment = project.experiments['hrpt']

Link the structural phase to the experiment.

In [13]:
experiment.linked_phases.create(id='lbco', scale=9.1351)

Set instrument and peak profile parameters.

These values provide the initial instrument description for the local
refinement. Later, a subset of them will be refined.

In [14]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.0

In [15]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1204
experiment.peak.broad_lorentz_y = 0.0844

Add background points and excluded regions.

The line-segment background is defined by a few anchor points. We also
exclude regions that are not intended to contribute to the fit.

In [16]:
experiment.background.create(id='1', x=10, y=168.5585)
experiment.background.create(id='2', x=30, y=164.3357)
experiment.background.create(id='3', x=50, y=166.8881)
experiment.background.create(id='4', x=110, y=175.4006)

In [17]:
experiment.excluded_regions.create(id='1', start=0, end=10)
experiment.excluded_regions.create(id='2', start=100, end=180)

## Step 4: Run an Initial Local Refinement

Before Bayesian sampling, it is useful to run a deterministic fit. This
gives us:

- a good point estimate near the best-fit region,
- uncertainties from the local optimizer,
- a quick check that the model and experiment are configured
  sensibly.

In this tutorial we refine only a small set of parameters that are easy
to interpret in the later Bayesian stage.

In [18]:
structure.cell.length_a.free = True

In [19]:
experiment.linked_phases['lbco'].scale.free = True
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.instrument.calib_twotheta_offset.free = True

We choose the BUMPS Levenberg-Marquardt minimizer as a fast local
optimizer. Its main purpose here is to provide a stable starting point
and uncertainty estimates for the Bayesian run.

In [20]:
project.analysis.minimizer.show_supported()

Minimizer types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,emcee,emcee affine-invariant ensemble Bayesian sampling
8,,lmfit,LMFIT library using the default Levenberg-Marquardt method
9,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
10,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [21]:
project.analysis.minimizer.type = 'bumps (lm)'

Current minimizer changed to


bumps (lm)


In [22]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

├── 📁 experiments/


│   └── 📄 hrpt.cif


├── 📁 analysis/


│   └── 📄 analysis.cif


└── 📄 summary.cif


In [23]:
project.display.fit.results()

⚙️ Settings used:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


...

,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),4.47
4,📏 Goodness-of-fit (reduced χ²),1.29
5,"📏 R-factor (Rf, %)",5.65
6,"📏 R-factor squared (Rf², %)",4.92
7,"📏 Weighted R-factor (wR, %)",4.08


📈 Refined parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8800,3.8913,0.0001,0.29 % ↑
2,hrpt,linked_phases,lbco,scale,,9.1351,9.1329,0.0333,0.02 % ↓
3,hrpt,peak,,broad_gauss_u,deg²,0.1000,0.0817,0.0078,18.33 % ↓
4,hrpt,peak,,broad_gauss_v,deg²,-0.1000,-0.1169,0.0057,16.91 % ↑
5,hrpt,instrument,,twotheta_offset,deg,0.0000,0.6306,0.0019,N/A


The correlation plot shows how strongly the fitted parameters move
together in the local refinement. The measured-vs-calculated plots show
how well the refined model reproduces the data globally and in a zoomed
region.

In [24]:
project.display.fit.correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
project.display.pattern(expt_name='hrpt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 5: Prepare for Bayesian Sampling

DREAM requires finite bounds for the free parameters. Instead of
setting them manually, we derive them from the uncertainties estimated
in the local refinement.

The helper method `set_fit_bounds_from_uncertainty` centers the bounds
on the current parameter value and expands them by a chosen multiple of
the reported uncertainty.

The default `multiplier` is 4. If the local refinement is very tight,
or if you expect a broader posterior, increase it explicitly.

Show unset fit bounds before setting them from the local refinement uncertainties.

In [26]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89134,0.00011,-inf,inf,Å
2,hrpt,linked_phases,lbco,scale,9.13288,0.03329,-inf,inf,
3,hrpt,peak,,broad_gauss_u,0.08167,0.00783,-inf,inf,deg²
4,hrpt,peak,,broad_gauss_v,-0.11691,0.00566,-inf,inf,deg²
5,hrpt,instrument,,twotheta_offset,0.63057,0.00191,-inf,inf,deg


Set fit bounds for all free parameters using the default multiplier of
4. In this tutorial that means the posterior pair plot will later
refer to a `±4 × uncertainty` region in its title. To use a different
region, pass another value, for example `multiplier=6`.

In [27]:
for param in project.free_parameters:
    param.set_fit_bounds_from_uncertainty()

Displaying the free parameters again is a convenient way to confirm
that the fit bounds have been assigned as expected before launching the
sampler.

In [28]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89134,0.00011,3.89091,3.89177,Å
2,hrpt,linked_phases,lbco,scale,9.13288,0.03329,8.99973,9.26603,
3,hrpt,peak,,broad_gauss_u,0.08167,0.00783,0.05034,0.11300,deg²
4,hrpt,peak,,broad_gauss_v,-0.11691,0.00566,-0.13955,-0.09427,deg²
5,hrpt,instrument,,twotheta_offset,0.63057,0.00191,0.62292,0.63823,deg


## Step 6: Configure and Run DREAM

We now switch from the local minimizer to the Bayesian DREAM sampler.

The settings below are intentionally small so the tutorial runs
quickly. For production analysis you would usually increase the number
of steps (`steps`) and often the burn-in (`burn`) as well. When
needed, the DREAM API also lets you tune how chains are initialized
through the `init` setting. Other sampler settings such as `thin` and
`pop` can be adjusted as well. The current EasyDiffraction defaults
use `steps=3000`, `init='lhs'`, and `parallel=0`, which tells
BUMPS-DREAM to use all available CPUs for population evaluations.

The `burn` setting is auto-resolved when left unset. With the default
`steps=3000` this gives `burn=600`, but if you override `steps` and
keep `burn=None`, the effective burn-in is recomputed automatically.
Here we use a much smaller step count to keep the tutorial fast, but
this is not recommended for production analysis.

In [29]:
project.analysis.minimizer.show_supported()

Minimizer types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,*,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,emcee,emcee affine-invariant ensemble Bayesian sampling
8,,lmfit,LMFIT library using the default Levenberg-Marquardt method
9,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
10,,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [30]:
project.analysis.minimizer.type = 'bumps (dream)'

⚠️ Switching minimizer type removes these settings:                                                                               
   • max_iterations                                                                                                               


⚠️ Switching minimizer type adds these settings with defaults:                                                                    
   • burn_in_steps=600                                                                                                            
   • initialization_method='latin_hypercube'                                                                                      
   • parallel_workers=0                                                                                                           
   • population_size=4                                                                                                            
   • random_seed=None                                                                                                             
   • sampling_steps=3000                                                                                                          
   • thinning_interval=1                                                           

Current minimizer changed to


bumps (dream)


In [31]:
project.analysis.minimizer.sampling_steps = 100  # lower than the default 3000
project.analysis.minimizer.burn_in_steps = 20  # lower than the default 600

In [32]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'bumps (dream)'...


📈 Bayesian sampling progress:


...

│   └── 📄 hrpt.cif


├── 📁 analysis/


│   ├── 📄 analysis.cif


│   └── 📄 results.h5


└── 📄 summary.cif


## Step 7: Inspect Bayesian Results

The fit-results display now includes sampler settings, convergence
diagnostics, committed parameter values, and posterior summary
statistics.

In [33]:
project.display.fit.results()

⚙️ Settings used:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Name,Value,Description
1,sampling_steps,100,Total sampler iterations per chain.
2,burn_in_steps,20,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,4,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,latin_hypercube,Sampler initialization method.
7,random_seed,None,Random seed; None uses a system-derived seed.


📋 Bayesian fit results:


...

,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↓
2,hrpt,linked_phases,lbco,scale,,9.1329,9.1329,0.0367,0.00 % ↓
3,hrpt,peak,,broad_gauss_u,deg²,0.0817,0.0817,0.0081,0.00 % ↓
4,hrpt,peak,,broad_gauss_v,deg²,-0.1169,-0.1169,0.0058,0.00 % ↓
5,hrpt,instrument,,twotheta_offset,deg,0.6306,0.6306,0.0019,0.00 % ↓


📊 Posterior distribution:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.235,169.9
2,hrpt,linked_phases,lbco,scale,,9.1367,"[9.0570, 9.2090]",1.298,137.6
3,hrpt,peak,,broad_gauss_u,deg²,0.0832,"[0.0642, 0.0974]",1.312,141.3
4,hrpt,peak,,broad_gauss_v,deg²,-0.1180,"[-0.1269, -0.1034]",1.312,133.1
5,hrpt,instrument,,twotheta_offset,deg,0.6306,"[0.6270, 0.6341]",1.243,157.6


The correlation and posterior-pair plots are complementary:

- `plot_param_correlations` summarizes pairwise structure in a compact
  matrix.
- `plot_posterior_pairs` shows marginal densities on the diagonal and
  posterior contours off-diagonal. In this tutorial its title also
  reminds you that the display region follows the `±4 × uncertainty`
  bounds defined above, while numeric subplot ranges are omitted to
  keep the grid readable.

In [34]:
project.display.fit.correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
project.display.posterior.pairs()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The one-dimensional posterior distributions below make it easier to
inspect individual parameters in isolation, including asymmetry or
multimodality.

In [36]:
project.display.posterior.distribution()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Finally, the posterior predictive plot propagates the sampled parameter
uncertainty into the calculated diffraction pattern. Comparing this to
the zoomed measured-vs-calculated view helps assess whether the sampled
model family explains the data in the region of interest.

In [37]:
project.display.posterior.predictive(expt_name='hrpt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

A final zoomed measured-vs-calculated plot is useful for checking how
the posterior-supported model behaves in a narrow region of the pattern
after the Bayesian run.

In [38]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>